In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import re
import pdfplumber
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import camelot
import googletrans
from googletrans import Translator
import datetime
from selenium import webdriver
from time import sleep
import os
import pdfplumber


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MN CBMONG' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running MN CBMONG Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://www.mongolbank.mn/en/p/1305',

        }



Typology={

       regulatorName + ' 1': 'National Payment System Licensed Entities',


        }

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

# Initialize the translator
translator = Translator()

def translate_text(text, retries=10):
    for i in range(retries):
        try:
            return translator.translate(text, src='mn', dest='en').text
        except Exception:
            if i == retries - 1:
                raise
            sleep(1.5)

def find_zip_code(string):
    match = re.search(r'\d{5}', string)
    if match:
        return match.group()
    else:
        return ''

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    driver.get(regdict[reg])
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    sleep(3)
    link = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "here")))
    link.click()
    sleep(3)
    tables = []
    print(os.listdir(tempfolder))
    pdf_file = os.listdir(tempfolder)[0]
    filePath = os.path.join(tempfolder, pdf_file)     
    with pdfplumber.open(filePath) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            tables.append(table)

    for table in tables:
        for tab in table:
            tab = [x for x in tab if x is not None]
            if tab[0].isdigit():
                # print(tab[0])
                name_ = tab[1].split('\n')[0]
                en_name_ = translate_text(name_)

                website_ = tab[1][tab[1].find('http'):]
                address_ = tab[2].replace('\n',' ')
                if address_:
                    en_address_ = translate_text(address_)
                else:
                    en_address_ = ''
                zip_code = find_zip_code(address_)
                if len(tab[1].split('\n'))>3:
                    
                    website_ = tab[1].split('\n')[1]
                    #print(website_ if len(website_)>5 else '')
                else:
                    website_ = tab[1][tab[1].find('http'):]
                    #print(website_ if len(website_)>5 else '')
                    
                sqldict['Name'].append(en_name_)
                sqldict['Name - Mother Company'].append(name_)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Address_1 - Mother company'].append(address_)
                sqldict['Address_1'].append(en_address_)
                sqldict['Website'].append(website_ if website_.startswith('http') else '')
                sqldict['Zip'].append(zip_code)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)

[INFO] : Working 1/1 _(MN CBMONG 1)_ 
['licenseLastUpdate20251217.pdf']


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)
for rem in os.listdir(tempfolder):
    os.remove(os.path.join(tempfolder, rem))